## 1. Imports and Configuration

In [88]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from datetime import datetime, timedelta


In [89]:
tickers = {
    "Information Technology": ["AAPL", "MSFT", "NVDA", "AVGO", "CRM"],  # High momentum, growth factor exposure
    "Financials": ["JPM", "BAC", "GS", "MS", "BLK", "WFC"],             # Value factor, rate sensitivity
    "Health Care": ["JNJ", "UNH", "LLY", "ABBV", "MRK", "PFE"],         # Defensive, quality factor
    "Consumer Discretionary": ["AMZN", "TSLA", "HD", "MCD", "NKE", "LOW"], # Cyclical, momentum variation
    "Industrials": ["CAT", "HON", "UNP", "RTX", "GE", "DE"],            # Classic value/quality mix
    "Communication Services": ["GOOGL", "META", "DIS", "NFLX", "T"],    # Growth vs. value spread
    "Consumer Staples": ["PG", "KO", "PEP", "WMT", "COST", "CL"],       # Low vol, defensive
    "Energy": ["XOM", "CVX", "COP", "SLB", "EOG"],                      # Value, commodity beta
    "Utilities": ["NEE", "DUK", "SO", "AEP", "EXC"],                    # Low vol, yield factor
    "Real Estate": ["PLD", "AMT", "EQIX", "SPG", "PSA"],                # Yield, rate sensitivity
    "Materials": ["LIN", "APD", "NEM", "FCX", "SHW"],                   # Cyclical, commodity exposure
}

all_tickers = [ticker for sector in tickers.values() for ticker in sector]

In [90]:
data = yf.download(
    [ticker for sector in tickers.values() for ticker in sector],
    start="2015-01-01",
    end="2025-01-01",
    interval = "1mo"
)["Close"]

C:\Users\godwi\AppData\Local\Temp\ipykernel_33088\2185564213.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(
[*********************100%***********************]  60 of 60 completed


In [91]:
print(data.shape)
# 120 Months x 60 stocks


(120, 60)


In [92]:
# Compute returns
returns = np.log(data).diff().dropna()
print(returns.shape)

(119, 60)


In [93]:
# Data Quality Checks
# 1. Are there any NaNs?
print(returns.isna().sum().sum())

# 2. Are they at the start?
print(returns.iloc[0].isna().sum())

# 3. Flag outliers (returns > 50% or < -50%)
outliers = (returns > 0.5) | (returns < -0.5)
outliers_df = returns[outliers].stack(level = 'Ticker')
print(outliers_df)


0
0
Date        Ticker
2016-02-01  FCX       0.506031
2020-03-01  EOG      -0.565959
            SLB      -0.682553
            SPG      -0.792845
2020-08-01  TSLA      0.554719
2022-04-01  NFLX     -0.676915
dtype: float64


In [94]:
data.to_csv("data/prices.csv")
returns.to_csv("data/returns.csv")

## 2. Data Loading, Universe Construction

 Load prices, compute returns, filter universe

In [95]:
# Compare to benchmark
spy_row = yf.download(
    "SPY",
    start="2015-01-01",
    end="2025-01-01",
    interval = "1mo"
)

spy_prices = spy_row["Close"]["SPY"]
spy_returns = np.log(spy_prices).diff().dropna()

print(spy_returns.isna().sum()) # Check
spy_returns.to_csv("data/spy_returns.csv")


C:\Users\godwi\AppData\Local\Temp\ipykernel_33088\1929563758.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  spy_row = yf.download(
[*********************100%***********************]  1 of 1 completed

0


In [96]:
# Pull Fundamental Data
# Market Cap
shares_dict = {}
for ticker in all_tickers:
    try:
        info = yf.Ticker(ticker).info
        shares = info.get('sharesOutstanding', None)
        shares_dict[ticker] = shares
    except Exception as e:
        print(f"Error fetching data for {ticker}: {e}")
        shares_dict[ticker] = None

shares_series = pd.Series(shares_dict)
missing = shares_series[shares_series.isna()]
print(missing) # None
log_market_cap = np.log(data.multiply(shares_series, axis=1))
log_market_cap.to_csv("data/log_market_cap.csv")

print(log_market_cap.shape) # 120 Months x 60 stocks
print(log_market_cap.head())



Series([], dtype: int64)
(120, 60)
                 AAPL       ABBV        AEP        AMT       AMZN        APD  \
Date                                                                           
2015-01-01  26.665469  24.928175  23.848913  24.259251  25.973893  23.849766   
2015-02-01  26.757546  24.938136  23.761974  24.281590  26.043693  23.919587   
2015-03-01  26.729611  24.905203  23.747128  24.229946  26.022263  23.887967   
2015-04-01  26.735381  25.004636  23.758090  24.233974  26.147584  23.839999   
2015-05-01  26.775554  25.042307  23.747839  24.219740  26.165093  23.862950   

                 AVGO        BAC        BLK        CAT  ...        SLB  \
Date                                                    ...              
2015-01-01  24.326491  25.163941  24.411277  24.043691  ...  25.222112   
2015-02-01  24.541985  25.206584  24.498166  24.087841  ...  25.243369   
2015-03-01  24.536958  25.179659  24.483028  24.052607  ...  25.240596   
2015-04-01  24.457024  25.217267  

In [102]:
!pip install statsmodels

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   ----- ---------------------------------- 1.3/9.5 MB 9.6 MB/s eta 0:00:01
   -------------- ------------------------- 3.4/9.5 MB 9.1 MB/s eta 0:00:01
   ----------------------- ---------------- 5.5/9.5 MB 9.1 MB/s eta 0:00:01
   ------------------------------ --------- 7.3/9.5 MB 9.1 MB/s eta 0:00:01
   -------------------------------------- - 9.2/9.5 MB 9.1 MB/s eta 0:00:01
   ---------------------------------------- 9.5/9.5 MB 8.9 MB/s eta 0:00:00



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: C:\Users\godwi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [103]:
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
# Check shape
print(returns.shape)        # (T, N)
print(spy_returns.shape)    # (T,)
# Confirm all share the same index
print(returns.index.equals(spy_returns.index))


(119, 60)
(119,)
True


In [114]:
# Compute Raw Signal Values
# 1. Market Beta
# Run OLS over past 36 months
beta_window = 36
monthly_index = returns.index
beta_panel = pd.DataFrame(np.nan, index = monthly_index, columns = all_tickers)

for i, date in enumerate(monthly_index):
    if i < beta_window:
        continue  # Not enough data for the first few months


    window_returns = returns.iloc[i-beta_window:i]
    window_spy = spy_returns.iloc[i-beta_window:i]
    
    valid_mask = window_spy.notna() 
    spy_window_clean = window_spy[valid_mask] # Not needed as already checked
    X = add_constant(spy_window_clean.values) # Add intercept -> Shape (35, 2) [1.0, x_t] Needed to multiply against \beta_0

    for ticker in all_tickers:
        y = window_returns.loc[window_returns.index[valid_mask], ticker]

        if y.isna().sum() > 3: 
            continue

        y_clean = y.fillna(0).values

        try:
            model = OLS(y_clean, X).fit()
            beta_panel.loc[date, ticker] = model.params[1]  # Store the beta coefficient

        except Exception:
            pass

beta_panel.to_csv("data/signal_beta_raw.csv")
print(beta_panel.shape)
    

(119, 60)


In [123]:
# Sanity check
print((beta_panel < -1).sum().sum())
print((beta_panel > 3).sum().sum()) 

0
0


## 3. Factor Signal Construction

Period-by-period WLS, store factor returns f_t

## 4. PCA on return Panel

### 4.1 Standardise return panel

### 4.2 PCA

### 4.3 Scree plot + Marchenko-Pastur cutoff

### 4.4 Extract loadings B and factor scores f_t

### 4.5 Compare PCA factors vs fundamental factors

## 5. Covariance Matrix Estimation

## 6. Validation and Diagnostics

IC, ICIR, t-stats, residual PCA check

## 7. Results and Visualisations